# Лабораторная 05. repartition, coalesce и output files

Цель: увидеть, как partitions влияют на shuffle и количество parquet-файлов.

In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName('lab-05-files').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
orders = spark.read.parquet((base / 'orders').as_uri())
out_base = base / 'lab05_output'
if out_base.exists():
    shutil.rmtree(out_base)
out_base.mkdir(parents=True, exist_ok=True)
print('Spark UI:', spark.sparkContext.uiWebUrl)

In [ ]:
def count_parquet_files(path):
    return len(list(Path(path).glob('*.parquet')))

def write_and_count(df, name):
    path = out_base / name
    df.write.mode('overwrite').parquet(path.as_uri())
    return count_parquet_files(path)


## Базовая запись

In [ ]:
orders.rdd.getNumPartitions(), write_and_count(orders, 'base')

## `repartition(20)`
Проверьте explain: должен появиться Exchange.

In [ ]:
orders_20 = orders.repartition(20)
orders_20.explain('formatted')
orders_20.rdd.getNumPartitions(), write_and_count(orders_20, 'repartition_20')

## `coalesce(2)`
Обычно дешевле, но может дать менее равномерные partitions.

In [ ]:
orders_2 = orders.coalesce(2)
orders_2.explain('formatted')
orders_2.rdd.getNumPartitions(), write_and_count(orders_2, 'coalesce_2')

Вопросы:

- Сколько файлов получилось в каждом каталоге?
- Почему количество файлов связано с partitions?
- Где был shuffle?
- Почему `repartition` дороже `coalesce`?
- Почему `coalesce(1)` может вызвать проблемы?